In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

In [2]:
model = load_model("age_gender_model.h5")
print("Model loaded successfully!")

Model loaded successfully!


In [3]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [4]:
# Load the Haar Cascade classifier for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

In [5]:
import cv2

In [6]:
cap = cv2.VideoCapture(0)
cap.set(3, 1280)   # width
cap.set(4, 720)    # height

cv2.namedWindow("Age & Gender Prediction", cv2.WINDOW_NORMAL)

age_buffer = []
gender_buffer = []
BUFFER_SIZE = 10

while True:
    ret, frame = cap.read()
    
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]

        try:
            face = cv2.resize(face, (64, 64))
            face = face / 255.0
            face = np.reshape(face, (1, 64, 64, 3))

            age_pred, gender_pred = model.predict(face, verbose=0)

            age = np.argmax(age_pred)
            gender = np.argmax(gender_pred)
            age_buffer.append(age)
            gender_buffer.append(gender)

            if len(age_buffer) > BUFFER_SIZE:
                age_buffer.pop(0)
                gender_buffer.pop(0)

            age = int(np.mean(age_buffer))
            gender = int(round(np.mean(gender_buffer)))

            gender = "Male" if gender == 0 else "Female"

            age_conf = int(np.max(age_pred) * 100)
            gender_conf = int(np.max(gender_pred) * 100)

            start = age * 10
            mid = start + 5

            if age_conf > 50:
                display_start = mid
                display_end = mid + 5
            else:
                display_start = start
                display_end = start + 5

            label = f"Age: {display_start}-{display_end} ({age_conf}%) | {gender} ({gender_conf}%)"

            cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0),2)
            cv2.putText(frame, label, (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

        except:
            continue

    cv2.imshow("Age & Gender Prediction", frame)

    # Press 'q' to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Exiting webcam...")
        break

cap.release()
cv2.destroyAllWindows()

Exiting webcam...
